# NIVAL airborne lidar versus NISAR dSWE, using SnowIn

This notebook adapts the NISAR comparison in Zach Hoppinen's [tc_brief_nival_validation](https://github.com/ZachHoppinen/tc_brief_nival_validation) repository to SnowIn. It targets Mores Creek Summit, Idaho, ASC-077, 7–19 February 2026. The reference analysis uses the operational NISAR L2 GUNW at 80 m and NIVAL lidar snow-depth rasters from 7 and 22 February.

The workflow removes the scene median phase before conversion; computes dSWE with SnowIn's canonical kernel, product wavelength and local incidence angle; differences the two lidar surfaces on their native grid; masks |dHS| ≥ 1.5 m there; average-resamples the gated change to the GUNW grid; anchors dSWE to median lidar-implied SWE at 147 kg/m³; and reports correlation, RMSE, fitted density and sample count by coherence gate. The phase datum removal happens before the angle-dependent conversion. The absolute SWE anchor is a separate constant offset that does not affect correlation. Zach's current code uses fixed wavelength 0.238 m; this notebook uses the GUNW metadata wavelength.

Reference README values for all / coherence >0.2 / >0.4 / >0.5 are n=4,976 / 2,891 / 1,004 / 463; r=0.82 / 0.93 / 0.96 / 0.96; RMSE=19 / 12 / 9 / 9 mm; and fitted density=150 / 157 / 161 / 162 kg/m³. The detailed stats script also includes >0.3, elevation deramping, scale decomposition, canopy/burn strata and local correlations, which require further masks and choices and are outside this minimal NISAR port.

Inputs are not bundled. The source repository identifies [NIVAL lidar at NSIDC](https://doi.org/10.5067/DPFDH2M49DQG) and the [NISAR L2 GUNW at ASF](https://doi.org/10.5067/NIL2GUNW-P1). Stage the named GUNW and two lidar rasters locally. The GUNW is about 2.5 GB. Supply a local-incidence raster aligned to the GUNW, in radians or degrees. SnowIn can calculate incidence through its geometry adapter, but this notebook leaves the DEM-dependent step explicit.

The correction screens carried in the GUNW are not applied, matching Zach's stated choice. Numerical differences can result from wavelength, LIA derivation, resampling, or valid-data masks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from snowin import compute_dswe
from snowin.io import open_gunw

GUNW = Path('/path/to/NISAR_L2_PR_GUNW_012_077_A_024_013_4000_SH_20260207T124619_20260207T124654_20260219T124619_20260219T124654_P05023_N_F_J_001.h5')
LIDAR_FEB07 = Path('/path/to/NIVAL_MCS_Lidar_20260207_SD_v01.tif')
LIDAR_FEB22 = Path('/path/to/NIVAL_MCS_Lidar_20260222_SD_v01.tif')
LOCAL_INCIDENCE = Path('/path/to/local_incidence_angle_on_gunw_grid.tif')
LIA_UNITS = 'radians'  # change to 'degrees' if that's what the raster stores
SNOW_DENSITY_KG_M3, WATER_DENSITY_KG_M3 = 147.0, 1000.0
DHS_GATE_M = 1.5
for path in (GUNW, LIDAR_FEB07, LIDAR_FEB22, LOCAL_INCIDENCE):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

In [ ]:
# SnowIn normalizes NISAR source phase to secondary_minus_reference.
pair = open_gunw(GUNW, chunks=None, progress=True)
phase, coherence = pair['phase'], pair['coherence']
print({'reference_time': pair.attrs.get('reference_time'), 'secondary_time': pair.attrs.get('secondary_time'), 'wavelength_m': pair.attrs.get('wavelength_m'), 'phase_convention': phase.attrs.get('phase_difference_definition'), 'shape': phase.shape, 'posting_m': float(abs(phase.x.values[1] - phase.x.values[0]))})

In [ ]:
lia_raster = rioxarray.open_rasterio(LOCAL_INCIDENCE, masked=True).squeeze()
if lia_raster.shape != phase.shape or not (np.allclose(lia_raster.y.values, phase.y.values) and np.allclose(lia_raster.x.values, phase.x.values)):
    raise ValueError('LIA raster must be exactly aligned with the GUNW phase grid')
lia = lia_raster.assign_coords(y=phase.y, x=phase.x)
if LIA_UNITS == 'degrees':
    lia = np.deg2rad(lia)
elif LIA_UNITS != 'radians':
    raise ValueError('LIA_UNITS must be radians or degrees')
lia = lia.rename('incidence_angle')
lia.attrs.update({'units': 'rad', 'incidence_angle_reference': 'local'})
phase_zeroed = (phase - phase.median(skipna=True)).rename('phase')
phase_zeroed.attrs = dict(phase.attrs)
phase_zeroed.attrs.update({'units': 'rad', 'phase_difference_definition': 'secondary_minus_reference'})
dswe_unanchored = compute_dswe(phase_zeroed, lia, wavelength_m=float(pair.attrs['wavelength_m']))
print('finite phase/dSWE pixels:', int(np.isfinite(dswe_unanchored.values).sum()))

In [ ]:
# Difference and gate dHS on the native lidar grid before average-resampling to 80 m,
# matching Zach's treatment of void/canopy outliers.
spatial_ref = pair['spatial_ref'].attrs
crs = spatial_ref.get('spatial_ref') or spatial_ref.get('crs_wkt')
if crs is None:
    raise ValueError('GUNW spatial_ref has no CRS text')
grid = xr.DataArray(np.zeros(phase.shape, dtype=np.float32), dims=phase.dims, coords={'y': phase.y, 'x': phase.x}, name='grid').rio.write_crs(crs)
def lidar_on_grid(path):
    raster = rioxarray.open_rasterio(path, masked=True).squeeze()
    return raster.rio.reproject_match(grid, resampling=Resampling.average)
depth_earlier = rioxarray.open_rasterio(LIDAR_FEB07, masked=True).squeeze()
depth_later = rioxarray.open_rasterio(LIDAR_FEB22, masked=True).squeeze()
if depth_earlier.shape != depth_later.shape or depth_earlier.rio.crs != depth_later.rio.crs or depth_earlier.rio.transform() != depth_later.rio.transform():
    raise ValueError('The two native lidar rasters must share a grid before differencing')
native_dhs = depth_later - depth_earlier
native_dhs = native_dhs.where((native_dhs > -DHS_GATE_M) & (native_dhs < DHS_GATE_M)).rio.write_nodata(np.nan)
dhs = native_dhs.rio.reproject_match(grid, resampling=Resampling.average).rename('dhs').assign_coords(y=phase.y, x=phase.x)
dhs.attrs.update({'units': 'm', 'interval': '2026-02-07 to 2026-02-22'})
common = np.isfinite(dswe_unanchored.values) & np.isfinite(dhs.values)
lidar_swe = dhs.values * SNOW_DENSITY_KG_M3 / WATER_DENSITY_KG_M3
anchor_m = np.nanmedian(lidar_swe[common] - dswe_unanchored.values[common])
dswe = (dswe_unanchored + anchor_m).rename('dswe')
print(f'common pixels: {int(common.sum()):,}; dSWE constant anchor: {anchor_m * 1000:.2f} mm')

In [ ]:
def comparison_metrics(mask=None):
    valid = common.copy()
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    n = int(valid.sum())
    if n < 3:
        return {'n': n, 'r': np.nan, 'rmse_mm': np.nan, 'density_kg_m3': np.nan}
    x, h = dswe.values[valid], dhs.values[valid]
    lidar = h * SNOW_DENSITY_KG_M3 / WATER_DENSITY_KG_M3
    return {'n': n, 'r': float(np.corrcoef(x, h)[0, 1]), 'rmse_mm': float(np.sqrt(np.mean((x - lidar) ** 2)) * 1000), 'density_kg_m3': float(np.polyfit(h, x, 1)[0] * WATER_DENSITY_KG_M3)}
gates = [('all pixels', None)] + [(f'coherence > {t:.1f}', coherence.values > t) for t in (0.2, 0.3, 0.4, 0.5)]
rows = []
for label, gate in gates:
    row = comparison_metrics(gate)
    row.update({'gate': label, 'percent_of_valid': 100 * row['n'] / int(common.sum())})
    rows.append(row)
pd.DataFrame(rows)[['gate', 'r', 'rmse_mm', 'density_kg_m3', 'n', 'percent_of_valid']]

## Interpretation and availability

The 147 kg/m³ anchor controls RMSE and fitted density, so it is an input assumption here rather than independently re-estimated. Confirm the exact LIA source, lidar nodata semantics and valid-pixel count before interpreting numerical differences. Zach states that correction screens carried in the GUNW are not applied; this notebook follows that choice.

Analysis code and pair census are in the linked GitHub repository. NIVAL lidar is distributed by NSIDC and the GUNW by ASF at the DOI links above. SNOTEL is available from NRCS. Freeman station measurements are available from the authors on request. The ancillary figures and other datasets are outside this NISAR-only port.